#IMP Operations in dataframes


##Sample DataFrame used in the examples below:


In [0]:
data = [
    (1, "Alice", "HR", 50000, "2021-01-15"),
    (2, "Bob", "Engineering", 85000, "2020-03-22"),
    (3, "Charlie", "Engineering", 95000, "2019-07-01"),
    (4, "David", "Sales", 60000, "2022-05-10"),
    (5, "Eva", None, 70000, "2021-11-30"),
]
columns = ["id", "name", "department", "salary", "join_date"]
df = spark.createDataFrame(data, columns)

##Choosing specific columns (like SQL SELECT).

In [0]:
df.select("name", "salary").show()

from pyspark.sql.functions import col
df.select(col("name"), col("salary") * 1.1).show()

df.select("*", (col("salary") / 12).alias("monthly_salary")).show()

##Identical functionality — where() is just a SQL-style alias for filter()


In [0]:
df.filter(col("salary") > 60000).show()
df.where(df["department"] == "Engineering").show()

# Multiple conditions
df.filter((col("salary") > 50000) & (col("department") == "Engineering")).show()
df.filter((col("department") == "HR") | (col("department") == "Sales")).show()

# NOT
df.filter(~(col("department") == "Engineering")).show()

#withColumn()

Add a new column or overwrite an existing one.

In [0]:
df2 = df.withColumn("bonus", col("salary") * 0.10)
df2 = df2.withColumn("department", col("department"))  # overwrite existing column

# Multiple columns at once (Spark 3.3+)
df3 = df.withColumns({
    "bonus": col("salary") * 0.10,
    "total_comp": col("salary") * 1.10
})

##drop()

In [0]:
df.drop("join_date").show()
df.drop("join_date", "department").show()

##distinct()

In [0]:
df.select("department").distinct().show()
df.dropDuplicates(["department"]).show()  # keep one row per department (any row)
df.dropDuplicates().show()                # remove fully duplicate rows

##orderBy() / sort()

Identical — sort() and orderBy() are aliases.

In [0]:
df.orderBy("salary").show()
df.orderBy(col("salary").desc()).show()
df.orderBy(col("department").asc(), col("salary").desc()).show()

##groupBy()

In [0]:
df.groupBy("department").count().show()
df.groupBy("department").sum("salary").show()
df.groupBy("department").avg("salary").show()

##agg()
More flexible than .sum()/.avg() — lets you compute multiple aggregates with custom names.

In [0]:
from pyspark.sql.functions import sum as _sum, avg, max as _max, min as _min, count

df.groupBy("department").agg(
    _sum("salary").alias("total_salary"),
    avg("salary").alias("avg_salary"),
    _max("salary").alias("max_salary"),
    _min("salary").alias("min_salary"),
    count("*").alias("employee_count"),
).show()